In [1]:
import torch
import sys
import os
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, Dataset
import torchxrayvision as xrv
import torchvision.transforms as transforms
from peft import LoraConfig, get_peft_model
import numpy as np
from PIL import Image
import pydicom
import torch.nn.functional as F
from tqdm import tqdm
import matplotlib.pyplot as plt

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(project_root)

from evaluation.utils.eval_utils import parse_pediatric_dataset, PediatricXRDataset
from machex.dataset_class.dataset import MaCheXDataset, ChestXrayDataset

d:\01 - Dokumente\01 - Studium\01 - StaDS Experience\01 - Kursmaterial\ADL - Applied Deep Learning\applied_dl\.venv_eval\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
base_path = r"D:\01 - Dokumente\01 - Studium\01 - StaDS Experience\01 - Kursmaterial\ADL - Applied Deep Learning\applied_dl\machex\chestx-ray\VinDr-PCXR\train"
save_root = r"D:\01 - Dokumente\01 - Studium\01 - StaDS Experience\01 - Kursmaterial\ADL - Applied Deep Learning\applied_dl\machex\torchxray_dataset_pediatric"
pathologies = ["No finding", "Pneumonia", "Bronchiolitis", "Brocho-pneumonia", "Tuberculosis"]

parse_pediatric_dataset(
    base_path=base_path,
    save_root=save_root,
    pathologies=pathologies,
    is_train=True
)

In [3]:
split='train'
base_path = r"D:\01 - Dokumente\01 - Studium\01 - StaDS Experience\01 - Kursmaterial\ADL - Applied Deep Learning\applied_dl\machex\torchxray_dataset_pediatric"

dataset = PediatricXRDataset(split, base_path)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=True)

In [ ]:
model = xrv.models.DenseNet(weights="densenet121-res224-all")

num_ftrs = model.classifier.in_features

model.classifier = torch.nn.Linear(num_ftrs, len(pathologies))

for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device) 

In [6]:
split='test'
base_path = r"D:\01 - Dokumente\01 - Studium\01 - StaDS Experience\01 - Kursmaterial\ADL - Applied Deep Learning\applied_dl\machex\torchxray_dataset_pediatric"

test_dataset = PediatricXRDataset(split, base_path)
test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=16, shuffle=True)

In [7]:
model.eval()
threshold = 0.5

# Initialize counters
TP = torch.zeros(len(pathologies)).to(device)
FP = torch.zeros(len(pathologies)).to(device)
FN = torch.zeros(len(pathologies)).to(device)
TN = torch.zeros(len(pathologies)).to(device)

with torch.no_grad():
    for inputs, labels in test_dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        # 1. Forward pass
        features = model.features(inputs)
        features = F.adaptive_avg_pool2d(features, (1, 1))
        features = torch.flatten(features, 1)
        logits = model.classifier(features)
        
        # 2. Thresholding
        predicted = (torch.sigmoid(logits) > threshold).float()
        
        # 3. Calculate statistics per class
        TP += (predicted * labels).sum(dim=0)
        FP += (predicted * (1 - labels)).sum(dim=0)
        FN += ((1 - predicted) * labels).sum(dim=0)
        TN += ((1 - predicted) * (1 - labels)).sum(dim=0)

# 4. Compute and Print Metrics
print(f"{'Pathology':<20} | {'Prec.':<7} | {'Recall':<7} | {'F1-Score':<7}")
print("-" * 55)

for i in range(len(pathologies)):
    precision = TP[i] / (TP[i] + FP[i] + 1e-9) # 1e-9 avoids division by zero
    recall = TP[i] / (TP[i] + FN[i] + 1e-9)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-9)
    
    print(f"{pathologies[i]:<20} | {precision.item():.4f} | {recall.item():.4f} | {f1.item():.4f}")

Pathology            | Prec.   | Recall  | F1-Score
-------------------------------------------------------
No finding           | 0.6328 | 0.9000 | 0.7431
Pneumonia            | 0.3333 | 0.1304 | 0.1875
Bronchiolitis        | 0.2700 | 0.9310 | 0.4186
Brocho-pneumonia     | 0.0732 | 0.3333 | 0.1200
Tuberculosis         | 0.0244 | 0.3333 | 0.0455


In [8]:
count = torch.zeros(len(pathologies)).to(device)

for i in range(len(test_dataset)):
    count += test_dataset[i][1].to(device)

print(count)

count_train = torch.zeros(len(pathologies)).to(device)

for i in range(len(dataset)):
    count_train += dataset[i][1].to(device)

print(count_train)

tensor([90., 23., 29.,  9.,  3.])
tensor([409.,  98.,  94.,  62.,  11.])


## Now with original Model (adult data)

In [28]:
split='test'
base_path = r"D:\01 - Dokumente\01 - Studium\01 - StaDS Experience\01 - Kursmaterial\ADL - Applied Deep Learning\applied_dl\machex\torchxray_dataset_pediatric"

# Example usage:
test_dataset = PediatricXRDataset(split, base_path)
test_dataloader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=True)

In [30]:
import torch
import torch.nn.functional as F
import torchxrayvision as xrv

# 1. Load the original base model
base_model = xrv.models.DenseNet(weights="densenet121-res224-all")
base_model.to(device)
base_model.eval()

# The 18 pathologies the model was originally trained on
base_targets = base_model.pathologies 

all_base_outputs = []
all_labels = []

with torch.no_grad():
    for images, labels in test_dataloader:
        images = images.to(device)
        
        # Standard forward pass (using the 18-class head)
        logits = base_model(images)
        
        # Apply sigmoid because XRV models output raw logits
        probs = torch.sigmoid(logits)
        
        all_base_outputs.append(probs.cpu())
        all_labels.append(labels.cpu())

# Stack results into large tensors
all_base_outputs = torch.cat(all_base_outputs) # Shape: [N, 18]
all_labels = torch.cat(all_labels)             # Shape: [N, 5]

In [32]:
threshold = 0.5

# 1. Identify the correct indices
# In torchxrayvision, "Pneumonia" is usually at index 8 or 12 depending on the version
# Let's find it dynamically:
base_pneu_idx = base_model.pathologies.index("Pneumonia")

# In your pediatric dataset: pathologies = ["No finding", "Pneumonia", ...]
# So Pneumonia is index 1
ped_pneu_idx = 1 

# 2. Extract the specific columns from our saved tensors
# (Assuming you cat'ed them into 'all_base_outputs' and 'all_labels' in the last step)
preds_probs = all_base_outputs[:, base_pneu_idx]
true_labels = all_labels[:, ped_pneu_idx]

# 3. Apply threshold
preds_binary = (preds_probs > threshold).float()

# 4. Calculate TP, FP, FN
tp = ((preds_binary == 1) & (true_labels == 1)).sum().item()
fp = ((preds_binary == 1) & (true_labels == 0)).sum().item()
fn = ((preds_binary == 0) & (true_labels == 1)).sum().item()

# 5. Compute Metrics
precision = tp / (tp + fp + 1e-9)
recall = tp / (tp + fn + 1e-9)
f1 = 2 * (precision * recall) / (precision + recall + 1e-9)

print(f"--- Base XRV Model vs. Pediatric Pneumonia ---")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

--- Base XRV Model vs. Pediatric Pneumonia ---
Precision: 0.1472
Recall:    0.9796
F1-Score:  0.2560


## Testing for BIM Covid Dataset

In [20]:
import numpy as np
import torch
import torchxrayvision as xrv

def xrv_classifier_transform(img):
    # 1. MaCheX loads images as RGB, but XRV DenseNet requires 1-channel Grayscale
    img = img.convert('L') 
    
    # 2. Convert to NumPy array
    arr = np.array(img)
    
    # 3. Add channel dimension for XRV [1, H, W]
    arr = arr[None, ...] 
    
    # 4. Apply XRV's specific crop and resize
    # We call them directly to ensure anatomical center-cropping
    arr = xrv.datasets.XRayCenterCrop()(arr)
    arr = xrv.datasets.XRayResizer(512)(arr)
    
    # 5. Normalize to [-1024, 1024] 
    # This is critical! XRV models will give wrong results with 0-1 scaling.
    arr = xrv.datasets.normalize(arr, 255)
    
    # 6. Return as tensor (The DataLoader will add the Batch dimension)
    return torch.from_numpy(arr).float()

# Re-instantiate your dataset and loader
dataset = MaCheXDataset(MACHEX_PATH, transforms=xrv_classifier_transform)

# Use batch_size=1 and num_workers=0 to ensure it runs smoothly on your laptop
dataloader = torch.utils.data.DataLoader(
    dataset, 
    batch_size=1, 
    shuffle=False, 
    num_workers=0, 
    collate_fn=lambda x: x[0] # Simplest way to handle the 'report' KeyError for inference
)

# Run Inference
model.eval()
with torch.no_grad():
    for batch in dataloader:
        # Since we use a simple collate_fn to avoid errors, batch is now one sample dict
        images = batch['img'].unsqueeze(0) # Add batch dimension manually here if batch_size=1
        preds = model(images)
        print(f"Pathology Probabilities: {dict(zip(model.pathologies, preds[0].cpu().numpy()))}")
        print('next one:')

Pathology Probabilities: {'Atelectasis': np.float32(0.17300995), 'Consolidation': np.float32(0.07627737), 'Infiltration': np.float32(0.42060846), 'Pneumothorax': np.float32(0.24746841), 'Edema': np.float32(0.02000815), 'Emphysema': np.float32(0.09307411), 'Fibrosis': np.float32(0.014251834), 'Effusion': np.float32(0.132797), 'Pneumonia': np.float32(0.2317001), 'Pleural_Thickening': np.float32(0.0475397), 'Cardiomegaly': np.float32(0.010318436), 'Nodule': np.float32(0.040442158), 'Mass': np.float32(0.018249027), 'Hernia': np.float32(0.00043150474), 'Lung Lesion': np.float32(0.5), 'Fracture': np.float32(0.29975256), 'Lung Opacity': np.float32(0.709376), 'Enlarged Cardiomediastinum': np.float32(0.5)}
next one:
Pathology Probabilities: {'Atelectasis': np.float32(0.090019256), 'Consolidation': np.float32(0.04196997), 'Infiltration': np.float32(0.20692818), 'Pneumothorax': np.float32(0.0123388525), 'Edema': np.float32(0.010030409), 'Emphysema': np.float32(0.05143907), 'Fibrosis': np.float32(

Exception ignored in: <function _ConnectionBase.__del__ at 0x0000019BB78F33A0>
Traceback (most recent call last):
  File "C:\Users\Karls\.pyenv\pyenv-win\versions\3.9.13\lib\multiprocessing\connection.py", line 137, in __del__
    self._close()
  File "C:\Users\Karls\.pyenv\pyenv-win\versions\3.9.13\lib\multiprocessing\connection.py", line 282, in _close
    _CloseHandle(self._handle)
OSError: [WinError 6] Das Handle ist ungültig


KeyboardInterrupt: 